# *Data Loading*

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("../data/dataset_with_weather_and_noise.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (25000, 44)


,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delayed,...,holiday_or_weekend_transit_flag,holiday_proximity_feature,api_temperature,api_humidity,api_wind_speed,precipitation,bad_weather_flag_api,temperature,humidity,wind_speed
0,250.99,amazon logistics,automobile parts,ev bike,standard,west,clear,244.719006,47.540697,no,...,0,1,33.122327,75.752202,17.619252,3.506765,0,32.483571,74.097849,17.056317
1,250.99,amazon logistics,clothing,bike,express,central,clear,85.025123,46.143220,yes,...,0,1,29.729640,95.602409,15.867221,2.943648,0,29.308678,94.778295,15.486040
2,250.99,amazon logistics,clothing,van,same day,north,clear,293.126234,31.971948,yes,...,1,8,33.545242,54.693465,11.853310,1.892765,0,33.238443,56.395895,13.558508
3,250.99,amazon logistics,cosmetics,ev bike,two day,central,stormy,107.903116,8.234676,no,...,0,2,37.927240,91.773339,8.088707,2.332062,1,37.615149,88.800189,7.938182
4,250.99,amazon logistics,cosmetics,ev van,two day,east,foggy,211.123548,7.186863,no,...,1,1,29.547111,72.695976,4.071856,1.645159,1,28.829233,72.374135,2.726783


In [4]:
safe_features = [
    "delivery_partner",
    "package_type",
    "vehicle_type",
    "delivery_mode",
    "region",
    "weather_condition",
    "distance_km",
    "package_weight_kg",
    "api_temperature",
    "api_humidity",
    "api_wind_speed",
    "bad_weather_flag_api",
    "holiday_or_weekend_transit_flag",
    "order_hour",
    "order_day",
    "is_weekend"
]

target = "delayed_flag_recon"

X = df[safe_features]
y = df[target]

In [5]:
import pandas as pd

X = pd.get_dummies(X, drop_first=True)

In [6]:
import joblib

joblib.dump(X.columns, "../models/classification_feature_columns.pkl")

['../models/classification_feature_columns.pkl']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())

print("After SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
delayed_flag_recon
0    19412
1      588
Name: count, dtype: int64
After SMOTE:
delayed_flag_recon
0    19412
1    19412
Name: count, dtype: int64


Random Forest Classifier

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_smote, y_train_smote)

rf_pred = rf_model.predict(X_test)

XGBoost Classifier

In [10]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb_model.fit(X_train_smote, y_train_smote)

xgb_pred = xgb_model.predict(X_test)

c:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:200: UserWarning: [16:50:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model Evaluation

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Random Forest Results")
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
print("Accuracy:", accuracy_score(y_test, rf_pred))


print("\nXGBoost Results")
print(confusion_matrix(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))
print("Accuracy:", accuracy_score(y_test, xgb_pred))

Random Forest Results
[[4820   33]
 [  45  102]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4853
           1       0.76      0.69      0.72       147

    accuracy                           0.98      5000
   macro avg       0.87      0.84      0.86      5000
weighted avg       0.98      0.98      0.98      5000

Accuracy: 0.9844

XGBoost Results
[[4819   34]
 [  26  121]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4853
           1       0.78      0.82      0.80       147

    accuracy                           0.99      5000
   macro avg       0.89      0.91      0.90      5000
weighted avg       0.99      0.99      0.99      5000

Accuracy: 0.988


In [12]:
results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred)
    ]
})

print(results)

           Model  Accuracy
0  Random Forest    0.9844
1        XGBoost    0.9880


| Model         | Precision | Recall   | F1       |
| ------------- | --------- | -------- | -------- |
| Random Forest | **0.83**  | 0.78     | 0.80     |
| XGBoost       | 0.79      | **0.86** | **0.82** |


XGBoost is better because:

Higher recall

Better F1-score

Chceking for overfitting :

In [13]:
from sklearn.metrics import accuracy_score

# Random Forest
rf_train_pred = rf_model.predict(X_train_smote)
rf_test_pred = rf_model.predict(X_test)

print("Random Forest Train Accuracy:", accuracy_score(y_train_smote, rf_train_pred))
print("Random Forest Test Accuracy:", accuracy_score(y_test, rf_test_pred))


# XGBoost
xgb_train_pred = xgb_model.predict(X_train_smote)
xgb_test_pred = xgb_model.predict(X_test)

print("\nXGBoost Train Accuracy:", accuracy_score(y_train_smote, xgb_train_pred))
print("XGBoost Test Accuracy:", accuracy_score(y_test, xgb_test_pred))

Random Forest Train Accuracy: 1.0
Random Forest Test Accuracy: 0.9844

XGBoost Train Accuracy: 0.998789408613229
XGBoost Test Accuracy: 0.988


In [14]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model, X, y, cv=5, scoring="accuracy")

print("Cross-validation scores:", scores)
print("Average CV accuracy:", scores.mean())

Cross-validation scores: [0.9844 0.9846 0.9854 0.9848 0.984 ]
Average CV accuracy: 0.98464


In [15]:
from sklearn.metrics import roc_auc_score

rf_probs = rf_model.predict_proba(X_test)[:,1]
xgb_probs = xgb_model.predict_proba(X_test)[:,1]

print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_probs))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb_probs))

Random Forest ROC-AUC: 0.9941743027316016
XGBoost ROC-AUC: 0.9966498035439192


Model Downlaoding:

In [16]:
import joblib
import os

# create models folder in parent directory
os.makedirs("../models", exist_ok=True)

# save models
joblib.dump(rf_model, "../models/random_forest_Classifier.pkl")
joblib.dump(xgb_model, "../models/xgboost_Classifier.pkl")

print("Classification models saved successfully!")

Classification models saved successfully!
